In [1]:
import pandas as pd
from pathlib import Path

In [3]:
model = "dinov3_only_CLS"
checkpoint = "facebook_dinov3-vith16plus-pretrain-lvd1689m"
step_intervals = [1,3,5]

In [4]:
files_path = Path("data/files")
sim_path = files_path / f"similarities_{model}-{checkpoint}.csv"
sim_df = pd.read_csv(sim_path)
sim_df = sim_df[sim_df['metric'] == 'cosine'].reset_index(drop=True)
sim_df.head(5)

,metric,query_folder,query_image,query_mod,instr_folder,instr_image,instr_mod,score
0,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_1.jpg,NaN,0.850379
1,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_17.jpg,NaN,0.768050
2,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_3.jpg,NaN,0.759884
3,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_2.jpg,NaN,0.758960
4,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_15.jpg,NaN,0.741114


In [5]:
image_df = pd.read_csv("data/files/image_records.csv")
image_df['modification'] = image_df['modification'].fillna('')

In [6]:
sim_df['query_mod'] = sim_df['query_mod'].fillna('')

In [7]:
sim_df['instr_mod'] = sim_df['instr_mod'].fillna('')

In [8]:
sim_df.head(5)

,metric,query_folder,query_image,query_mod,instr_folder,instr_image,instr_mod,score
0,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_1.jpg,,0.850379
1,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_17.jpg,,0.768050
2,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_3.jpg,,0.759884
3,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_2.jpg,,0.758960
4,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_15.jpg,,0.741114


In [9]:
grouped = sim_df.groupby(['query_folder','query_image','query_mod'])

In [ ]:
step_lookup = image_df.set_index(['folder_path', 'image_name', 'modification'])['step_number']

best = sim_df.loc[grouped['score'].idxmax()].copy()
best['query_step'] = pd.MultiIndex.from_frame(
    best[['query_folder', 'query_image', 'query_mod']]).map(step_lookup)
best['instr_step'] = pd.MultiIndex.from_frame(
    best[['instr_folder', 'instr_image', 'instr_mod']]).map(step_lookup)
for interval in step_intervals:
    hf_interval = interval // 2
    best[f'{interval}_step_match'] = ((best['instr_step'] >= (best['query_step'] - hf_interval)) & (best['instr_step'] <= (best['query_step'] + hf_interval))).astype(float)    

In [ ]:
best = best.rename(columns={'instr_folder': 'most_similar_instr_folder',
                     'instr_image': 'most_similar_instr_image',
                     'instr_mod': 'most_similar_instr_mod'})
step_match_path = files_path / f"step_match_{model}-{checkpoint}.csv"
best.to_csv(step_match_path, sep=',', index=False)

# Aggregating scores by category

In [24]:
def images_by_build(buildName: str):
    records = pd.read_csv(files_path / "image_records.csv")
    records['modification'] = records['modification'].fillna('')
    return records.loc[records['build_name'] == buildName]


def images_with_tags(tags: list[str]):
    records = pd.read_csv(files_path / "image_records.csv")
    records['modification'] = records['modification'].fillna('')
    record_tags = records['tags'].fillna('').str.split('|').apply(set)
    return records.loc[record_tags.apply(set(tags).issubset)]

In [25]:
output_df = pd.DataFrame(
    columns=['Categories','1_step_match_prtg','3_step_match_prtg','5_step_match_prtg']
)

In [26]:
def aggregate_step_match_scores_as_percentages(df:pd.DataFrame, intervals: list[int]) -> pd.DataFrame:
    row = {}
    for interval in intervals:
        col = f'{interval}_step_match'
        row[f'{interval}_step_match_prtg'] = round(df[col].mean() * 100, 2) if col in df.columns else float('nan')
    return pd.DataFrame([row])


In [27]:
step_match_df = pd.read_csv(step_match_path)
step_match_df['query_mod'] = step_match_df['query_mod'].fillna('')

all = aggregate_step_match_scores_as_percentages(step_match_df, step_intervals)
all.insert(0, 'Categories', 'All dataset')
output_df = pd.concat([output_df, all], ignore_index=True)

In [28]:
for build in ['45401-B_build-1','45401-B_build-2','45401-B_build-3']:
    images_by_build_df = images_by_build(build)
    idx = pd.MultiIndex.from_frame(images_by_build_df[['folder_path', 'image_name', 'modification']])
    mask = pd.MultiIndex.from_frame(step_match_df[['query_folder', 'query_image', 'query_mod']]).isin(idx)
    step_match_build_df = step_match_df[mask].copy()
    build_tot_scores = aggregate_step_match_scores_as_percentages(step_match_build_df, step_intervals)
    build_tot_scores.insert(0,'Categories',build)
    output_df = pd.concat([output_df, build_tot_scores], ignore_index=True)

In [29]:
for tag in ['reference','-90angle','top','left_side_light','low_light','white_background','wooden_background']:
    for kind in ['clean','dirty']:
        images_by_tags_df = images_with_tags([tag,kind])
        idx = pd.MultiIndex.from_frame(images_by_tags_df[['folder_path', 'image_name', 'modification']])
        mask = pd.MultiIndex.from_frame(step_match_df[['query_folder', 'query_image', 'query_mod']]).isin(idx)
        step_match_tags_df = step_match_df[mask].copy()
        tags_category_tot_scores = aggregate_step_match_scores_as_percentages(step_match_tags_df, step_intervals)
        name_string = '_'.join([tag,kind])
        tags_category_tot_scores.insert(0,'Categories',name_string)
        output_df = pd.concat([output_df, tags_category_tot_scores], ignore_index=True)

In [30]:
output_df

,Categories,1_step_match_prtg,3_step_match_prtg,5_step_match_prtg
0,All dataset,53.887195,76.371951,82.77439
1,45401-B_build-1,55.952381,87.698413,93.253968
2,45401-B_build-2,64.60396,83.168317,87.871287
3,45401-B_build-3,46.493902,67.835366,75.609756
4,reference_clean,63.157895,86.315789,89.473684
5,reference_dirty,46.315789,70.526316,78.947368
6,-90angle_clean,68.085106,90.425532,93.617021
7,-90angle_dirty,50.0,71.276596,77.659574
8,top_clean,44.680851,69.148936,76.595745
9,top_dirty,31.914894,48.93617,57.446809


In [32]:
for kind in ['clean','dirty']:
    images_by_tags_df = images_with_tags([kind])
    idx = pd.MultiIndex.from_frame(images_by_tags_df[['folder_path', 'image_name', 'modification']])
    mask = pd.MultiIndex.from_frame(step_match_df[['query_folder', 'query_image', 'query_mod']]).isin(idx)
    step_match_tags_df = step_match_df[mask].copy()
    tags_category_tot_scores = aggregate_step_match_scores_as_percentages(step_match_tags_df, step_intervals)
    name_string = '_'.join([kind])
    tags_category_tot_scores.insert(0,'Categories',name_string)
    output_df = pd.concat([output_df, tags_category_tot_scores], ignore_index=True)

In [33]:
output_df

,Categories,1_step_match_prtg,3_step_match_prtg,5_step_match_prtg
0,All dataset,53.887195,76.371951,82.77439
1,45401-B_build-1,55.952381,87.698413,93.253968
2,45401-B_build-2,64.60396,83.168317,87.871287
3,45401-B_build-3,46.493902,67.835366,75.609756
4,reference_clean,63.157895,86.315789,89.473684
5,reference_dirty,46.315789,70.526316,78.947368
6,-90angle_clean,68.085106,90.425532,93.617021
7,-90angle_dirty,50.0,71.276596,77.659574
8,top_clean,44.680851,69.148936,76.595745
9,top_dirty,31.914894,48.93617,57.446809


In [35]:
output_df.to_csv(f"tmp_csv/step_match_{model}-{checkpoint}.csv")